In [23]:
"""
validation_02.py — 跨資料集驗證腳本 (notebooks/ 版本)
═════════════════════════════════════════════════════════════════════════════
適用資料夾結構:
  專案根目錄/
  ├── bias_engine.py                          ← 在父層
  └── notebooks/
      ├── validation_02.py                    ← 本檔案
      └── data/
          ├── cleaned_hr_data.csv             ← 你的主資料
          └── validation/
              ├── 1.inx_future.csv            ← 動作 3 必要
              └── 2.rich_hr.csv               ← 動作 1 補強

執行方式:
  cd notebooks/
  python validation_02.py

【動作 1】跨資料集驗證    - 在 INX、Rich's HR 上驗證框架通用性
【動作 2】Baseline 對照   - 在主資料上跑 Naive / Linear / Ridge / GBM / RF
【動作 3】形成性 vs 反映性 - 在主資料 + INX 上跑構念效度比對
═════════════════════════════════════════════════════════════════════════════
"""

"\nvalidation_02.py — 跨資料集驗證腳本 (notebooks/ 版本)\n═════════════════════════════════════════════════════════════════════════════\n適用資料夾結構:\n  專案根目錄/\n  ├── bias_engine.py                          ← 在父層\n  └── notebooks/\n      ├── validation_02.py                    ← 本檔案\n      └── data/\n          ├── cleaned_hr_data.csv             ← 你的主資料\n          └── validation/\n              ├── 1.inx_future.csv            ← 動作 3 必要\n              └── 2.rich_hr.csv               ← 動作 1 補強\n\n執行方式:\n  cd notebooks/\n  python validation_02.py\n\n【動作 1】跨資料集驗證    - 在 INX、Rich's HR 上驗證框架通用性\n【動作 2】Baseline 對照   - 在主資料上跑 Naive / Linear / Ridge / GBM / RF\n【動作 3】形成性 vs 反映性 - 在主資料 + INX 上跑構念效度比對\n═════════════════════════════════════════════════════════════════════════════\n"

In [24]:
import warnings; warnings.filterwarnings("ignore")
import os
import sys
import numpy as np
import pandas as pd

In [25]:
# ── bias_engine 在父層,加入路徑(支援 Jupyter %run 與直接執行兩種模式)──
try:
    _here = os.path.dirname(os.path.abspath(__file__))
except NameError:                       # Jupyter / IPython 環境沒有 __file__
    _here = os.getcwd()
sys.path.insert(0, os.path.abspath(os.path.join(_here, "..")))
sys.path.insert(0, os.path.abspath(".."))

In [26]:
# ── 中文字型設定(若要產生含中文圖表時用) ──
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
_zh_fonts = ['Noto Sans CJK TC', 'PingFang TC', 'Heiti TC',
             'Microsoft JhengHei', 'Microsoft YaHei',
             'Arial Unicode MS', 'DejaVu Sans']
_installed = {f.name for f in fm.fontManager.ttflist}
_use = next((f for f in _zh_fonts if f in _installed), 'DejaVu Sans')
matplotlib.rcParams['font.sans-serif'] = [_use] + _zh_fonts
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.unicode_minus'] = False

In [27]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.metrics import r2_score
from bias_engine import HRSchemaMapper, AutoFeatureEngineer, GeneralizedBiasEngine

In [28]:
OUT_DIR = "validation_02_output"
os.makedirs(OUT_DIR, exist_ok=True)
LOG_LINES = []

def banner(t):
    line = "═" * 78
    msg = f"\n{line}\n  {t}\n{line}"
    print(msg); LOG_LINES.append(msg)

def log(s=""):
    print(s); LOG_LINES.append(str(s))

In [29]:
# ═════════════════════════════════════════════════════════════════════════════
# 資料集路徑
# ═════════════════════════════════════════════════════════════════════════════
DATASETS = {
    "main": {
        "label":   "【本研究】HRDataset_v2",
        "path":    "data/cleaned_hr_data.csv",
        "target":  "ManagerRating",
        "sensitive": ["Gender", "MaritalStatus", "Ethnicity"],
        "objective": ["YearsAtCompany", "OverTime", "TrainingOpportunitiesTaken",
                       "X_EngagementRate", "Experience_SEM",
                       "T_ManagerRating", "T_JobSatisfaction", "T_WorkLifeBalance"],
        "satisfaction_items": ["EnvironmentSatisfaction", "JobSatisfaction",
                                "RelationshipSatisfaction", "WorkLifeBalance"],
        "group_col": "EmployeeID",
    },
    "inx": {
        "label":   "INX Future Inc (IABAC)",
        "path":    "data/validation/1.inx_future.csv",
        "target":  "PerformanceRating",
        "sensitive": ["Gender", "MaritalStatus", "EducationBackground"],
        "objective": ["Age", "DistanceFromHome", "EmpEducationLevel", "EmpHourlyRate",
                       "NumCompaniesWorked", "EmpLastSalaryHikePercent",
                       "TotalWorkExperienceInYears", "TrainingTimesLastYear",
                       "ExperienceYearsAtThisCompany", "ExperienceYearsInCurrentRole",
                       "YearsSinceLastPromotion", "YearsWithCurrManager",
                       "EmpJobInvolvement", "EmpJobLevel"],
        "satisfaction_items": ["EmpEnvironmentSatisfaction", "EmpJobSatisfaction",
                                "EmpRelationshipSatisfaction", "EmpWorkLifeBalance"],
        "group_col": "EmpNumber",
    },
    "rich": {
        "label":   "Rich Huebner's HR Data (real, academic)",
        "path":    "data/validation/2.rich_hr.csv",
        "target":  "PerfScoreID",
        "sensitive": ["Sex", "RaceDesc", "MaritalDesc"],
        "objective": ["Age", "Salary", "EngagementSurvey", "EmpSatisfaction",
                       "SpecialProjectsCount", "DaysLateLast30", "Absences"],
        "satisfaction_items": None,
        "group_col": "EmpID",
    },
}

In [30]:
# ═════════════════════════════════════════════════════════════════════════════
# 共用函式
# ═════════════════════════════════════════════════════════════════════════════
def cronbach_alpha(items_df):
    k = items_df.shape[1]
    return (k/(k-1)) * (1 - items_df.var(ddof=1).sum() / items_df.sum(axis=1).var(ddof=1))

def load(cfg):
    if not os.path.exists(cfg["path"]):
        return None
    df = pd.read_csv(cfg["path"])
    # 部分資料集薪資欄位含 $ 與千分位(如 Rich Huebner)
    for c in df.columns:
        if df[c].dtype == "object":
            sample = df[c].dropna().astype(str).head(20).tolist()
            if any("$" in s for s in sample):
                df[c] = (df[c].astype(str).str.replace(r"[\$,]", "", regex=True)
                              .replace({"": np.nan}).astype(float))
    return df

def encode_for_baseline(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns and not pd.api.types.is_numeric_dtype(out[c]):
            out[c] = LabelEncoder().fit_transform(out[c].astype(str))
    return out

In [31]:
# ═════════════════════════════════════════════════════════════════════════════
# 動作 2:Baseline 對照
# ═════════════════════════════════════════════════════════════════════════════
def action_2_baseline(cfg):
    banner(f"動作 2｜Baseline 對照 — {cfg['label']}")
    df = load(cfg)
    if df is None:
        log(f"  ✗ 找不到 {cfg['path']},跳過。"); return None

    OBJ = [c for c in cfg["objective"] if c in df.columns]
    log(f"  資料形狀: {df.shape}    可用客觀特徵: {len(OBJ)} 項")

    work = encode_for_baseline(df, OBJ)
    X = work[OBJ].values
    y = work[cfg["target"]].astype(float).values
    if cfg.get("group_col") and cfg["group_col"] in work.columns:
        groups = work[cfg["group_col"]].values
        cv = GroupKFold(n_splits=5)
        cv_kwargs = {"cv": cv, "groups": groups, "n_jobs": -1}
    else:
        cv = KFold(n_splits=5, shuffle=True, random_state=42)
        cv_kwargs = {"cv": cv, "n_jobs": -1}

    models = {
        "Naive (預測平均)":      DummyRegressor(strategy="mean"),
        "Linear Regression":     LinearRegression(),
        "Ridge Regression":      Ridge(alpha=1.0, random_state=42),
        "Gradient Boosting":     GradientBoostingRegressor(n_estimators=200, max_depth=5, random_state=42),
        "Random Forest (本研究)": RandomForestRegressor(n_estimators=200, max_depth=10,
                                                      random_state=42, n_jobs=-1),
    }
    rows = []
    for name, m in models.items():
        cvp = cross_val_predict(m, X, y, **cv_kwargs)
        r2 = r2_score(y, cvp)
        rows.append({"模型": name, "CV R²": round(r2, 4)})
    rdf = pd.DataFrame(rows)
    base = rdf[rdf["模型"]=="Linear Regression"]["CV R²"].iloc[0]
    rdf["相對 Linear 提升 (pp)"] = ((rdf["CV R²"] - base)*100).round(2)

    log("\n" + rdf.to_string(index=False))
    rf_ = rdf[rdf["模型"]=="Random Forest (本研究)"]["CV R²"].iloc[0]
    log(f"\n  → RF vs Linear: 絕對提升 {(rf_-base)*100:.2f} pp,"
        f"相對提升 {(rf_-base)/max(base,1e-6)*100:.1f}%")

    name = os.path.basename(cfg["path"]).replace(".csv", "")
    rdf.to_csv(f"{OUT_DIR}/表_Baseline_{name}.csv", index=False, encoding="utf-8-sig")
    return rdf

In [32]:
# ═════════════════════════════════════════════════════════════════════════════
# 動作 3:形成性 vs 反映性
# ═════════════════════════════════════════════════════════════════════════════
def action_3_formative(cfg):
    banner(f"動作 3｜形成性 vs 反映性 — {cfg['label']}")
    df = load(cfg)
    if df is None:
        log(f"  ✗ 找不到 {cfg['path']},跳過。"); return None
    items = cfg.get("satisfaction_items")
    if not items:
        log(f"  ✗ 此資料集滿意度子維度不足,不做動作 3。"); return None
    items = [c for c in items if c in df.columns]
    if len(items) < 3:
        log(f"  ✗ 可用滿意度欄位僅 {len(items)} 項(< 3),不做動作 3。"); return None
    log(f"  滿意度子維度({len(items)} 項): {items}")

    data = df[items].dropna().astype(float)

    a = cronbach_alpha(data)
    flag = '✗ 未達 0.7 (Nunnally, 1978)' if a < 0.7 else '✓ 達 0.7'
    log(f"\n  【反映性測量】Cronbach's α = {a:.4f}    {flag}")

    Xc = sm.add_constant(data.values)
    log(f"\n  【形成性測量】VIF (含常數項;Petter et al., 2007 形成性門檻 < 3.3)")
    vif_rows = []
    for i, c in enumerate(["(常數)"] + items):
        v = variance_inflation_factor(Xc, i)
        vif_rows.append({"指標": c, "VIF": round(v, 3)})
        log(f"    {c:35s} VIF = {v:.3f}")
    vif_df = pd.DataFrame(vif_rows)

    corr = data.corr().round(3)
    log(f"\n  【兩兩相關矩陣】")
    log(corr.to_string())

    if cfg["target"] in df.columns:
        common = df[items + [cfg["target"]]].dropna()
        y = common[cfg["target"]].astype(float).values
        X = common[items].astype(float).values
        lr = LinearRegression().fit(X, y)
        log(f"\n  【形成性外部效度】(指標 → {cfg['target']})")
        for c, coef in zip(items, lr.coef_):
            log(f"    β({c:35s}) = {coef:+.4f}")
        log(f"    截距 = {lr.intercept_:.4f}    R² = {lr.score(X, y):.4f}")

    name = os.path.basename(cfg["path"]).replace(".csv", "")
    vif_df.to_csv(f"{OUT_DIR}/表_VIF_{name}.csv", index=False, encoding="utf-8-sig")
    corr.to_csv(f"{OUT_DIR}/表_相關_{name}.csv", encoding="utf-8-sig")
    return {"alpha": float(round(a, 4)),
            "vif_range": f"{vif_df.iloc[1:]['VIF'].min():.3f}–{vif_df.iloc[1:]['VIF'].max():.3f}",
            "corr_range": f"{corr.values[np.triu_indices(len(items), k=1)].min():.3f}"
                          f"–{corr.values[np.triu_indices(len(items), k=1)].max():.3f}"}

In [33]:
# ═════════════════════════════════════════════════════════════════════════════
# 動作 1:跨資料集驗證
# ═════════════════════════════════════════════════════════════════════════════
def action_1_run_pipeline(cfg):
    banner(f"動作 1｜框架通用性 — {cfg['label']}")
    df = load(cfg)
    if df is None:
        log(f"  ✗ 找不到 {cfg['path']},跳過。"); return None
    if cfg["target"] not in df.columns:
        log(f"  ✗ 找不到目標欄位 {cfg['target']},跳過。"); return None

    SENS = [c for c in cfg["sensitive"] if c in df.columns]
    OBJ  = [c for c in cfg["objective"] if c in df.columns]
    log(f"  形狀={df.shape}  目標={cfg['target']}  敏感={SENS}  客觀數={len(OBJ)}")
    if len(OBJ) < 3:
        log(f"  ✗ 可用客觀特徵不足(< 3),跳過。"); return None

    try:
        mapper = HRSchemaMapper({"target": cfg["target"], "sensitive": SENS, "objective": OBJ})
        clean = mapper.validate_and_clean(df)
        proc, fobj = AutoFeatureEngineer().fit_transform(clean, mapper.objective)
        engine = GeneralizedBiasEngine(mapper.target, fobj, mapper.sensitive)
        gc = cfg.get("group_col")
        res = engine.run(proc, method="residual",
                         group_col=gc if gc and gc in df.columns else None)
        cv_r2 = r2_score(clean[cfg["target"]].astype(float), res["Objective_Rating"])
        vc = res["Bias_Flag"].value_counts()
        out = {"資料集": cfg["label"], "N": len(df), "CV R²": round(cv_r2, 4),
                "Fair": int(vc.get("Fair", 0)),
                "向上校正": int(vc.get("Suggest Higher", 0)),
                "向下校正": int(vc.get("Suggest Lower", 0))}
        if SENS:
            try:
                gf = engine.run(proc, method="group_fairness", sensitive_target=SENS[0])
                d = gf["Disparate_Impact"]
                out[f"DI 範圍 ({SENS[0]})"] = f"{min(d.values()):.2f}–{max(d.values()):.2f}"
            except Exception:
                pass
        log(f"  ✓ CV R² = {cv_r2:.4f}    偏誤標記: {dict(vc)}")
        return out
    except Exception as e:
        log(f"  ✗ Pipeline 失敗:{e}")
        return None

In [34]:
# ═════════════════════════════════════════════════════════════════════════════
# 主程式
# ═════════════════════════════════════════════════════════════════════════════
def main():
    banner("論文 90+ 增值套件 — validation_02 (notebooks/ 版本)")
    log(f"工作目錄: {os.getcwd()}")

    available = {k: cfg for k, cfg in DATASETS.items() if os.path.exists(cfg["path"])}
    log(f"\n[資料集偵測]")
    for k, cfg in DATASETS.items():
        log(f"  {('✓' if k in available else '✗')} {cfg['label']:40s} {cfg['path']}")

    if "main" not in available:
        log(f"\n❌ 主資料 {DATASETS['main']['path']} 不存在,無法執行。")
        log(f"   請確認你已在 notebooks/ 資料夾下執行,且 data/cleaned_hr_data.csv 存在。")
        return

    banner("【執行】動作 2:Baseline 對照")
    baseline_main = action_2_baseline(DATASETS["main"])
    baseline_inx  = action_2_baseline(DATASETS["inx"]) if "inx" in available else None

    banner("【執行】動作 3:形成性 vs 反映性構念分析")
    fa_main = action_3_formative(DATASETS["main"])
    fa_inx  = action_3_formative(DATASETS["inx"]) if "inx" in available else None

    banner("【執行】動作 1:跨資料集通用性驗證")
    pipeline_results = []
    for k in ["main", "inx", "rich"]:
        if k in available:
            r = action_1_run_pipeline(DATASETS[k])
            if r: pipeline_results.append(r)

    banner("彙整:跨資料集比較表")
    if len(pipeline_results) >= 1:
        df1 = pd.DataFrame(pipeline_results)
        log(df1.to_string(index=False))
        df1.to_csv(f"{OUT_DIR}/表_跨資料集比較.csv", index=False, encoding="utf-8-sig")

    if fa_main or fa_inx:
        banner("彙整:構念效度跨資料集比較(動作 3)")
        rows = []
        if fa_main: rows.append({"資料集": DATASETS["main"]["label"], **fa_main})
        if fa_inx:  rows.append({"資料集": DATASETS["inx"]["label"],  **fa_inx})
        if rows:
            df3 = pd.DataFrame(rows)
            log(df3.to_string(index=False))
            log("\n  → 解讀:若兩份資料集 α 都偏低、VIF 都很低,")
            log("    則「四項滿意度為形成性構念」之主張具跨資料集穩健性,")
            log("    動作 3 的學術論述強度倍增。")
            df3.to_csv(f"{OUT_DIR}/表_構念效度比較.csv", index=False, encoding="utf-8-sig")

    with open(f"{OUT_DIR}/執行紀錄.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(LOG_LINES))
    banner("完成")
    log(f"輸出位置: notebooks/{OUT_DIR}/")
    log(f"產物: Baseline、VIF、相關矩陣、跨資料集比較、構念效度比較等 csv")


if __name__ == "__main__":
    main()


══════════════════════════════════════════════════════════════════════════════
  論文 90+ 增值套件 — validation_02 (notebooks/ 版本)
══════════════════════════════════════════════════════════════════════════════
工作目錄: /Users/belindaty.yu/Desktop/碩士/論文/論文資料/HRDataset_v2/HR_Bias_Platform(0524)/notebooks

[資料集偵測]
  ✓ 【本研究】HRDataset_v2                        data/cleaned_hr_data.csv
  ✓ INX Future Inc (IABAC)                   data/validation/1.inx_future.csv
  ✓ Rich Huebner's HR Data (real, academic)  data/validation/2.rich_hr.csv

══════════════════════════════════════════════════════════════════════════════
  【執行】動作 2:Baseline 對照
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
  動作 2｜Baseline 對照 — 【本研究】HRDataset_v2
══════════════════════════════════════════════════════════════════════════════
  資料形狀: (1224, 47)    可用客觀特徵: 8 項

                 模型   CV R²  相對 Linear 提升 (pp)
       Nai